# Notebook 01 — Data Exploration

Loads and explores the synthetic biometric score distributions used throughout the MSc biometric authentication analysis (CETM44).

The three CSV files represent:
- **High accuracy** (EER ≈ 1%) — a state-of-the-art system
- **Medium accuracy** (EER ≈ 5%) — an average commercial system
- **Low accuracy** (EER ≈ 15%) — a legacy or degraded system

**GDPR Note:** Synthetic score distributions only — no real biometric data (Art.5(1)(c) data minimisation).


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sns.set_theme(style='whitegrid', palette='muted')
DATA_DIR = Path('../data/synthetic')

high   = pd.read_csv(DATA_DIR / 'scores_high_accuracy.csv')
medium = pd.read_csv(DATA_DIR / 'scores_medium_accuracy.csv')
low    = pd.read_csv(DATA_DIR / 'scores_low_accuracy.csv')

print('High accuracy:  ', high.shape)
print('Medium accuracy:', medium.shape)
print('Low accuracy:   ', low.shape)
high.head()


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=False)
datasets = [('High accuracy (EER≈1%)', high), ('Medium accuracy (EER≈5%)', medium), ('Low accuracy (EER≈15%)', low)]

for ax, (title, df) in zip(axes, datasets):
    genuine  = df[df['label'] == 1]['score']
    impostor = df[df['label'] == 0]['score']
    ax.hist(impostor, bins=60, alpha=0.6, color='#C8102E', label='Impostor', density=True)
    ax.hist(genuine,  bins=60, alpha=0.6, color='#007A33', label='Genuine',  density=True)
    ax.set_title(title)
    ax.set_xlabel('Comparison Score')
    ax.set_ylabel('Density')
    ax.legend()

plt.suptitle('Genuine vs Impostor Score Distributions\n(Overlapping Gaussian model, Daugman 2003)', y=1.02)
plt.tight_layout()
plt.show()


In [ ]:
results = []
for name, df in [('high', high), ('medium', medium), ('low', low)]:
    for cls, cls_name in [(1, 'genuine'), (0, 'impostor')]:
        s = df[df['label'] == cls]['score']
        results.append({'dataset': name, 'class': cls_name, 'n': len(s),
                        'mean': s.mean(), 'std': s.std(),
                        'min': s.min(), 'max': s.max()})
pd.DataFrame(results).round(4)


In [ ]:
print(high.groupby(['group', 'label']).size().unstack(fill_value=0))
